Monte carlo simulatie voor de depressiegraaf

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit


In [ ]:
#initializing the graph, hard coded from the board
#we use the connection matrix defined during the yap session
"""
1: Concentratie
2: Traag/Zenuwachtig
3: Suicidaal
4: Laag zelfbeeld
5: Weinig interesse
6: Neerslachtig
7: Slechte slaap
8: Moe
9: Eetstoornis

We doen verbindings_matrix[i,j] is van i naar j
"""
verbindings_matrix = [[0,0,0,0,0,0,0,0,0], 
                      [,,,,,,,,], 
                      [,,,,,,,,], 
                      [,,,,,,,,], 
                      [,,,,,,,,], 
                      [,,,,,,,,], 
                      [,,,,,,,,], 
                      [,,,,,,,,], 
                      [,,,,,,,,]]

extern_veld = [,,,,,,,,]

In [ ]:
# Monte carlo simulatie met credits naar prof. Enrico Carlon
def initialize_lattice(L):
    """Initialize an LxL lattice with random spins (+1 or -1)."""
    return np.random.choice([-1, 1], size=(L))
@njit
def compute_energy(lattice, parameters):
    """Compute the total energy of the lattice."""
    intern = parameters[0]
    extern = parameters[1]
    energy = 0
    L = lattice.shape[0]
    for i in range(L):
        external = lattice[i]*extern[i]
        internal = 0
        for j in range(L):
            internal += lattice[i]*lattice[j]*intern[i,j]
        energy += external + internal            
    return energy # Each pair counted twice
@njit
def metropolis_step(lattice, beta, dEs, probs):
    """Perform one Metropolis update step."""
    L = lattice.shape[0]
    dE = 0
    for _ in range(L):
        i = np.random.randint(0, L, 2)
        S = lattice[i, j]
        neighbors = [lattice[(i+1)%L,j], lattice[i,(j+1)%L],lattice[(i-1)%L,j],lattice[i,(j-1)%L]].count(lattice[i,j])
        if dEs[neighbors] < 0 or np.random.rand() < probs[neighbors]:
            lattice[i, j] *= -1
            dE += dEs[neighbors]
    return dE
def simulate_ising(L, T, sweeps, equilib_sweeps):
    """Simulate the Ising model and compute average energy & energy squared"""
    beta = 1.0 / T
    lattice = initialize_lattice(L)
### Precompute an array with the possible values of the energy change
### and Metropolis probabilities
    dEs = np.array((-8, -4, 0, 4, 8))
    probs = np.exp(-beta * dEs)
    probs[:2] = 1
    E = compute_energy(lattice)
    for _ in range(equilib_sweeps):
        E += metropolis_step(lattice, beta, dEs, probs)
    E_total=E;E2_total=E**2
    for _ in range(sweeps):
        E += metropolis_step(lattice, beta, dEs, probs)
        E_total += E
        E2_total += E**2
    E_avg=E_total/sweeps;E2_avg=E2_total/sweeps
    return E_avg, E2_avg
def compute_specific_heat(L, T_values, sweeps, equilib_sweeps=100):
    """Compute specific heat over a range of temperatures."""
    C_V_values = []
    for T in T_values:
    # print(f"T: {T:.2f}")
        E_avg, E2_avg = simulate_ising(L, T, sweeps, equilib_sweeps)
        C_V = (E2_avg - E_avg**2) / (T**2 * L**2)
        C_V_values.append(C_V)
    return C_V_values
### Simulation Parameters
L = 9 # Lattice size
T_values = np.linspace(1.5, 3.5, 20) # Temperature range
sweeps = 10000 # Number of Monte Carlo sweeps
equilib_sweeps = 100 # Number of equilibration sweeps
# Compute Specific Heat
C_V_values = compute_specific_heat(L, T_values, sweeps, equilib_sweeps)
# print('C_V_values:', C_V_values)
# Plot the Results
plt.figure(figsize=(8, 5))
plt.plot(T_values, C_V_values, 'bo-', label='Specific Heat')
plt.xlabel('Temperature (T)')
plt.ylabel('Specific Heat ($C_V$)')
plt.title('Specific Heat vs Temperature in 2D Ising Model')
plt.legend()
plt.grid()
plt.show()